# Post 010 — Hyperparameter Optimization
## Dataset A: 3D Printer Quality Optimization

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

A 3D printer has dozens of tunable parameters: extrusion temperature, print speed, layer height, cooling fan speed, retraction distance, and more. Set them wrong and you get stringing, layer separation, or blobs. Set them right and you get a perfect print.

Manually tuning these parameters takes days of trial and error. This notebook demonstrates **Grid Search**, **Randomized Search**, and **Bayesian Optimization (Optuna)** to find the optimal parameter combination — and shows why Bayesian optimization finds better results in far fewer trials.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import optuna
import time
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/printer_quality_optimization.csv')
print(f'Shape: {df.shape}')
print(f'\nTarget distribution (print quality score 0-100):')
print(df['quality_score'].describe())
df.head()

## 1. Understanding the Optimization Problem

We have a dataset of 5,000 print experiments, each with different parameter settings and a measured quality score. Our goal is to:
1. Train a surrogate model (GradientBoostingRegressor) to predict quality from parameters
2. Use three search strategies to find the best hyperparameters for the surrogate model
3. Compare how many trials each strategy needs to reach the same quality level

This is a meta-optimization: we're optimizing the optimizer.

In [ ]:
feature_cols = [c for c in df.columns if c != 'quality_score']
X = df[feature_cols].values
y = df['quality_score'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Visualize quality distribution and feature correlations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(y, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(x=y.mean(), color='red', linestyle='--', label=f'Mean: {y.mean():.1f}')
ax1.set_title('Print Quality Score Distribution')
ax1.set_xlabel('Quality Score (0-100)'); ax1.set_ylabel('Count')
ax1.legend()

corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax2, 
            square=True, linewidths=0.5, annot_kws={'size': 8})
ax2.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 2. Strategy 1: Grid Search

Grid Search exhaustively tries every combination of parameter values. With 3 parameters and 4 values each, that's 4³ = 64 combinations. It guarantees finding the best combination within the grid, but the search space explodes exponentially with more parameters.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1, 0.2]
}
total_combinations = 4 * 4 * 4
print(f'Grid Search: {total_combinations} combinations × 5-fold CV = {total_combinations * 5} model fits')

start = time.time()
grid_search = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid, 
                           cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train_s, y_train)
grid_time = time.time() - start

y_pred_grid = grid_search.predict(X_test_s)
print(f'\nBest params: {grid_search.best_params_}')
print(f'Test MAE: {mean_absolute_error(y_test, y_pred_grid):.3f}')
print(f'Test R²: {r2_score(y_test, y_pred_grid):.3f}')
print(f'Time: {grid_time:.1f}s')

## 3. Strategy 2: Randomized Search

Instead of trying every combination, Randomized Search samples randomly from the parameter distributions. With the same number of trials as Grid Search, it often finds better results because it can explore a much wider range of values — including non-integer values that Grid Search would never try.

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(2, 8),
    'learning_rate': uniform(0.005, 0.295)
}

start = time.time()
random_search = RandomizedSearchCV(GradientBoostingRegressor(random_state=42), param_dist,
                                   n_iter=64, cv=5, scoring='neg_mean_absolute_error',
                                   random_state=42, n_jobs=-1)
random_search.fit(X_train_s, y_train)
random_time = time.time() - start

y_pred_random = random_search.predict(X_test_s)
print(f'Best params: {random_search.best_params_}')
print(f'Test MAE: {mean_absolute_error(y_test, y_pred_random):.3f}')
print(f'Test R²: {r2_score(y_test, y_pred_random):.3f}')
print(f'Time: {random_time:.1f}s')

## 4. Strategy 3: Bayesian Optimization (Optuna)

Bayesian Optimization is fundamentally different. Instead of searching blindly, it:
1. Builds a **probabilistic surrogate model** (TPE — Tree-structured Parzen Estimator) of the objective function
2. Uses an **acquisition function** (Expected Improvement) to decide where to sample next
3. Updates the surrogate model after each trial

Result: it learns from previous trials and focuses on promising regions. Same number of trials, significantly better results.

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20)
    }
    model = GradientBoostingRegressor(**params, random_state=42)
    from sklearn.model_selection import cross_val_score
    scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -scores.mean()

start = time.time()
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=64, show_progress_bar=False)
bayesian_time = time.time() - start

best_model = GradientBoostingRegressor(**study.best_params, random_state=42)
best_model.fit(X_train_s, y_train)
y_pred_bayes = best_model.predict(X_test_s)

print(f'Best params: {study.best_params}')
print(f'Test MAE: {mean_absolute_error(y_test, y_pred_bayes):.3f}')
print(f'Test R²: {r2_score(y_test, y_pred_bayes):.3f}')
print(f'Time: {bayesian_time:.1f}s')

## 5. The Key Comparison: Convergence Speed

The most important visualization: how quickly does each method find a good solution? Bayesian Optimization should reach its best result in fewer trials.

In [ ]:
# Extract convergence curves
grid_scores = [-s for s in grid_search.cv_results_['mean_test_score']]
random_scores = [-s for s in random_search.cv_results_['mean_test_score']]
bayes_scores = [t.value for t in study.trials]

grid_best = [min(grid_scores[:i+1]) for i in range(len(grid_scores))]
random_best = [min(random_scores[:i+1]) for i in range(len(random_scores))]
bayes_best = [min(bayes_scores[:i+1]) for i in range(len(bayes_scores))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

ax1.plot(range(1, len(grid_best)+1), grid_best, color='steelblue', label='Grid Search', linewidth=2)
ax1.plot(range(1, len(random_best)+1), random_best, color='coral', label='Random Search', linewidth=2)
ax1.plot(range(1, len(bayes_best)+1), bayes_best, color='green', label='Bayesian (Optuna)', linewidth=2)
ax1.set_title('Convergence: Best MAE Found vs Number of Trials')
ax1.set_xlabel('Trial Number'); ax1.set_ylabel('Best MAE Found')
ax1.legend()

# Final comparison bar chart
methods = ['Grid Search', 'Random Search', 'Bayesian (Optuna)']
maes = [mean_absolute_error(y_test, y_pred_grid),
        mean_absolute_error(y_test, y_pred_random),
        mean_absolute_error(y_test, y_pred_bayes)]
r2s = [r2_score(y_test, y_pred_grid),
       r2_score(y_test, y_pred_random),
       r2_score(y_test, y_pred_bayes)]
colors_bar = ['steelblue', 'coral', 'green']

bars = ax2.bar(methods, maes, color=colors_bar, alpha=0.8)
ax2.set_title('Final Test MAE Comparison (same 64 trials each)')
ax2.set_ylabel('Test MAE (lower is better)')
for bar, mae, r2 in zip(bars, maes, r2s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'MAE={mae:.3f}\nR²={r2:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'Grid Search:    MAE={maes[0]:.3f}, R²={r2s[0]:.3f}')
print(f'Random Search:  MAE={maes[1]:.3f}, R²={r2s[1]:.3f}')
print(f'Bayesian Opt:   MAE={maes[2]:.3f}, R²={r2s[2]:.3f}')

## 6. Summary

| Method | Search Strategy | Parameter Space | Trials Needed | Best For |
|---|---|---|---|---|
| **Grid Search** | Exhaustive | Discrete grid | High | Small search spaces, guaranteed coverage |
| **Random Search** | Random sampling | Continuous distributions | Medium | Wide search spaces, quick baseline |
| **Bayesian (Optuna)** | Probabilistic, adaptive | Continuous + discrete | Low | Expensive evaluations, large search spaces |

**Key insight**: Bayesian Optimization is not just faster — it explores a richer parameter space (continuous values, log-scale learning rates, additional parameters like `subsample`) and finds better solutions because it learns from its own history.